<a href="https://colab.research.google.com/github/r-minhyuk/deep-learning-utils/blob/main/Resnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from torch import nn

class BasicBlock(nn.Module):
  expansion=1

  def __init__(self, inplanes, planes, stride=1):
    super().__init__()
    self.conv1=nn.Conv2d(
        inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False
    )
    self.bn1=nn.BatchNorm2d(planes)
    self.relu=nn.ReLU(inplace=True)
    self.conv2=nn.Conv2d(
        planes, planes, kernel_size=3, stride=1, padding=1, bias=False
    )
    self.bn2=nn.BatchNorm2d(planes)

    self.shortcut=nn.Sequential()
    if stride!=1 or inplanes !=self.expansion*planes:
      self.shortcut=nn.Sequential(
          nn.Conv2d(
              inplanes,
              self.expansion*planes,
              kernel_size=1,
              stride=stride,
              bias=False
          ),
          nn.BatchNorm2d(self.expansion*planes)
          )


  def forward(self, x):
    out=self.conv1(x)
    out=self.bn1(out)
    out=self.relu(out)
    out=self.conv2(out)
    out=self.bn2(out)
    out+=self.shortcut(x)
    out=self.relu(out)
    return out

In [2]:
class BottleneckBlock(nn.Module):
  expansion=4

  def __init__(self, inplanes, planes, stride=1):
    super().__init__()
    self.conv1=nn.Conv2d(
       inplanes, planes, kernel_size=1, stride=stride, bias=False
    )
    self.bn1=nn.BatchNorm2d(planes)
    self.conv2=nn.Conv2d(
        planes, planes, kernel_size=3, stride=1, padding=1, bias=False
    )
    self.bn2=nn.BatchNorm2d(planes)
    self.conv3=nn.Conv2d(
        planes, planes*self.expansion, kernel_size=1, bias=False
    )
    self.bn3=nn.BatchNorm2d(planes*self.expansion)
    self.relu=nn.ReLU(inplace=True)
    self.shortcut=nn.Sequential()
    if stride !=1 or inplanes != self.expansion*planes:
      self.shortcut=nn.Sequential(
          nn.Conv2d(
              inplanes,
              self.expansion*planes,
              kernel_size=1,
              stride=stride,
              bias=False
          ),
          nn.BatchNorm2d(self.expansion*planes)
      )


  def forward(self, x):
    out=self.conv1(x)
    out=self.bn1(out)
    out=self.relu(out)
    out=self.conv2(out)
    out=self.bn2(out)
    out=self.relu(out)
    out=self.conv3(out)
    out=self.bn3(out)
    out+=self.shortcut(x)
    out=self.relu(out)
    return out

In [3]:
#레즈넷 모델 클래스
#ResNet 18/34는 BasicBottle, 50/101/152는 BottleneckBlock 이용
import torch

class ResNet(nn.Module):
  def __init__(self, block, layers, num_classes=1000):
    super().__init__()

    self.inplanes=64
    self.stem=nn.Sequential(
        nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False),
        nn.BatchNorm2d(self.inplanes),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    )
    self.stage1=self._make_layer(block, 64, layers[0], stride=1)
    self.stage2=self._make_layer(block, 128, layers[1], stride=2)
    self.stage3=self._make_layer(block, 256, layers[2], stride=2)
    self.stage4=self._make_layer(block, 512, layers[3], stride=2)

    self.avgpool=nn.AdaptiveAvgPool2d((1,1))
    self.fc=nn.Linear(512*block.expansion, num_classes)

  def _make_layer(self, block, planes, num_blocks, stride):
    layers=[]
    layers.append(block(self.inplanes, planes, stride))
    self.inplanes=planes*block.expansion
    for _ in range(num_blocks-1):
      layers.append(block(self.inplanes, planes,1))

    return nn.Sequential(*layers)


  def forward(self, x):
    out=self.stem(x)
    out=self.stage1(out)
    out=self.stage2(out)
    out=self.stage3(out)
    out=self.stage4(out)
    out=self.avgpool(out)
    out=torch.flatten(out, 1)
    out=self.fc(out)
    return out

In [4]:
!pip install torchinfo

In [5]:
#ResNet 모델 비교해보기
import torchvision
import torchinfo

from torchvision import models
from torchinfo import summary

resnet34=ResNet(BasicBlock, [3,4,6,3])
torch_model=models.resnet34(weights='ResNet34_Weights.IMAGENET1K_V1')

resnet34_info=summary(resnet34, (1,3,224,224))
torch_model_info=summary(torch_model, (1,3,224,224))

print(resnet34_info.total_params)
print(torch_model_info.total_params)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 143MB/s]


21797672
21797672
